# fdriver 1단계 시범 분석 v2 (위약 검정 보완)

v1 리뷰에서 나온 문제: 위약 hs 상관을 기업의 매출 전체 기간에서 한 번만 계산해, 표본이 짧은 드라이버
(Nanya 25분기, 항공 14분기, 아마존 30분기 등)가 긴 기간 위약과 비교됐다. 표본이 짧을수록 상관이 크게 나오기 쉬워
짧은 드라이버가 과대평가된다.

v2 에서 바꾼 것
1. 위약 hs 상관을 드라이버 x 시차마다, 그 드라이버 상관을 계산한 것과 같은 분기들에서 계산한다. 추세 가짜 시계열도 같은 분기 (trend_n 으로 확인)
2. 위약 hs 수 30 -> 200 (기업별 고정 시드, 같은 4자리 품목군 제외). 그 분기들에 YoY 가 모두 있는 후보를 섞은 순서대로 200개. 실제 수는 n_placebo
3. 판정 기준은 v1 과 같고, 위약 백분위 97.5 이상을 `강함` 으로 따로 표시
4. 발표 시점 우위: 같은 분기 (매출 available_at - 드라이버 available_at) 일수 중앙값 (시차 0)
5. v1 과 v2 판정 비교. 원인을 나누려고 "위약 200개 + 매출 전체 기간"(v1 방식) 판정도 함께 계산
6. 결과물은 git 밖 OUTPUT_ROOT/stage1_pilot/v2 에 저장. 위약 띠는 드라이버별로 그린다

1단계 목적은 흐름 검증이며, 상관은 참고용이다. DB 는 읽기만 한다.

In [ ]:
import sys, logging, warnings
from pathlib import Path
here = Path.cwd().resolve()
ROOT = next(p for p in [here, *here.parents] if (p / "DATA" / "fdriver").is_dir())   # .../investment_strategy
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.ERROR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from DATA.fdriver import asof, db, entity, panel as fp
from DATA.fdriver.config import output_path, output_root
from DATA.fdriver.adapters import trading_calendar
from DATA.fdriver.pilots import pilot_analysis_v2 as pa

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 400)
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 차트 색: 기본 검증 팔레트 범주색 1~3번(고정 순서), 회색 참조 띠, 글자색
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
BAND, INK, INK2, GRID, SURFACE = "#dcdad5", "#0b0b0b", "#52514e", "#e9e8e4", "#fcfcfb"

FIG_DIR = output_path("stage1_pilot", "v2", "figures")
TAB_DIR = output_path("stage1_pilot", "v2", "tables")
V1_TAB = output_path("stage1_pilot", "v1", "tables", mkdir=False)
PILOTS = entity.PILOTS
WRITES_AT_START = len(db.WRITE_LOG)
print("OUTPUT_ROOT:", output_root())
print("figures:", FIG_DIR); print("tables:", TAB_DIR)

## 1. 거래일 달력과 panel

In [ ]:
cal = trading_calendar()
print(f"거래일 {len(cal):,}일, {cal.min().date()} ~ {cal.max().date()}")
info = entity.entity_info(list(PILOTS)).set_index("entity_id")
panels, wides, yoys, avails = {}, {}, {}, {}
for eid in PILOTS:
    p = fp.panel(eid, trade_items=pa.TRADE_ITEMS.get(eid, ("expDlr", "expWgt")))
    panels[eid] = p
    wides[eid] = fp.panel_wide(p)
    yoys[eid] = pa.yoy(wides[eid])
    avails[eid] = p.pivot(index="period_end", columns="driver", values="available_at")
summary = pd.DataFrame([{"종목": e, "기업": PILOTS[e], "panel 행": len(p), "시계열 수": p["driver"].nunique(),
                         "매출 YoY 분기 수": int(yoys[e]["revenue"].notna().sum()),
                         "A": round((p["asof_quality"] == "A").mean(), 3), "B": round((p["asof_quality"] == "B").mean(), 3),
                         "C": round((p["asof_quality"] == "C").mean(), 3)} for e, p in panels.items()])
display(summary)

In [ ]:
# as_of 재확인 (실제 panel, 기업별 무작위 날짜 10개)
chk = []
for eid, p in panels.items():
    rng = np.random.default_rng(int(eid))
    for d in pd.to_datetime(rng.integers(p["available_at"].min().value, p["available_at"].max().value, 10)).normalize():
        got = asof.as_of(p, d)
        chk.append({"종목": eid, "날짜": d.date(), "미래 값": int((got["available_at"] > d).sum()),
                    "잘못 버린 값": int((p["available_at"] <= d).sum()) - len(got)})
asof_check = pd.DataFrame(chk)
assert asof_check["미래 값"].sum() == 0 and asof_check["잘못 버린 값"].sum() == 0
asof_check.to_csv(TAB_DIR / "asof_check.csv", index=False, encoding="utf-8-sig")
print("as_of 검사", len(asof_check), "건, 미래 값", asof_check["미래 값"].sum(), "건")

## 2. 위약 hs 준비

후보: 수출 월 데이터 200개월 이상인 6자리 hs 중, 그 기업에 연결된 hs 와 같은 4자리 품목군을 뺀 것.
후보 전체를 기업별 고정 시드(종목코드)로 섞고, 드라이버 x 시차마다 그 분기들을 모두 채운 후보를 섞은 순서대로 200개 쓴다.

In [ ]:
pool = pa.placebo_pool()
orders, plcs = {}, {}
rows = []
for eid in PILOTS:
    order = pa.placebo_order(eid, pool)
    orders[eid] = order
    plcs[eid] = pa.placebo_yoy(order, int(info.loc[eid, "fiscal_year_end"]), yoys[eid].index)
    rows.append({"종목": eid, "기업": PILOTS[eid], "전체 후보 풀": len(pool), "제외 후 후보": len(order),
                 "YoY 계산된 후보": plcs[eid].shape[1]})
display(pd.DataFrame(rows))

## 3. 상관과 위약 검정 (v2: 같은 분기 위약 200개)

In [ ]:
res_v2, res_full = {}, {}
for eid in PILOTS:
    drivers = pa.ordered_drivers(eid, wides[eid].columns)
    res_v2[eid] = pa.evaluate(yoys[eid], drivers, plcs[eid], avails[eid], scope="same")
    res_full[eid] = pa.evaluate(yoys[eid], drivers, plcs[eid], avails[eid], scope="full")
all_v2 = pd.concat([r.assign(entity=e, name=PILOTS[e]) for e, r in res_v2.items()], ignore_index=True)
all_full = pd.concat([r.assign(entity=e, name=PILOTS[e]) for e, r in res_full.items()], ignore_index=True)
all_v2.to_csv(TAB_DIR / "corr_placebo_v2_all.csv", index=False, encoding="utf-8-sig")
all_full.to_csv(TAB_DIR / "corr_placebo_full200_all.csv", index=False, encoding="utf-8-sig")
assert (all_v2["trend_n"] == all_v2["n"]).all(), "추세 상관이 다른 분기에서 계산됨"
print("행 수", len(all_v2), "/ 추세 분기 수 = 드라이버 분기 수: 모든 행에서 같음")
print("n_placebo 분포:", all_v2["n_placebo"].describe()[["min", "50%", "max"]].to_dict())

KOR = {"driver": "드라이버", "lag": "선행(분기)", "corr": "상관", "n": "표본", "n_placebo": "위약 수",
       "placebo_pct_abs": "위약 백분위", "placebo_p05": "위약 5%", "placebo_p95": "위약 95%",
       "trend_corr": "추세 상관", "verdict": "판정", "strong": "강함", "timing_adv_days": "발표 우위(일)",
       "start": "시작", "end": "끝"}
COLS = ["driver", "lag", "corr", "n", "n_placebo", "placebo_pct_abs", "placebo_p05", "placebo_p95",
        "trend_corr", "verdict", "strong", "timing_adv_days", "start", "end"]
def show(df):
    display(df[COLS].rename(columns=KOR).round(3))

In [ ]:
for eid in PILOTS:
    display(Markdown(f"### {eid} {PILOTS[eid]}"))
    r = res_v2[eid]
    show(r[r["driver"].isin(pa.SPEC_DRIVERS[eid])])

In [ ]:
# 판정 집계 (명세서 후보 드라이버 x 시차)
spec_v2 = all_v2[[d in pa.SPEC_DRIVERS[e] for d, e in zip(all_v2["driver"], all_v2["entity"])]]
tally = spec_v2.pivot_table(index=["entity", "name"], columns="verdict", values="corr", aggfunc="size", fill_value=0)
tally["강함(97.5 이상)"] = spec_v2.groupby(["entity", "name"])["strong"].sum()
tally.to_csv(TAB_DIR / "verdict_tally_v2.csv", encoding="utf-8-sig")
display(tally)

### 발표 시점 우위 (시차 0)
같은 분기 기준 (매출 available_at - 드라이버 available_at) 일수의 중앙값. 양수면 드라이버를 매출보다 그만큼 먼저 알 수 있다.
매출은 분기 45일, 4분기 90일(법정 기한, C), 수출입은 분기 마지막 달 + 15일이라 대체로 30일 앞선다.

In [ ]:
timing = (spec_v2[spec_v2["lag"] == 0][["name", "driver", "timing_adv_days", "corr", "verdict", "strong"]]
          .rename(columns=KOR))
timing.to_csv(TAB_DIR / "timing_advantage_v2.csv", index=False, encoding="utf-8-sig")
display(timing)

## 4. v1 과 v2 판정 비교

In [ ]:
v1 = pd.read_csv(V1_TAB / "corr_placebo_all.csv", dtype={"entity": str})
v1["entity"] = v1["entity"].str.zfill(6)
rev_n = {e: int(yoys[e]["revenue"].notna().sum()) for e in PILOTS}
cmp_ = pa.compare_v1(v1, all_v2, all_full, rev_n)
cmp_["name"] = cmp_["entity"].map(PILOTS)
cmp_.to_csv(TAB_DIR / "v1_vs_v2_verdict.csv", index=False, encoding="utf-8-sig")
changed = cmp_[cmp_["changed"]].sort_values(["entity", "driver", "lag"])
print(f"전체 {len(cmp_)}행 중 판정이 바뀐 행 {len(changed)}")
display(changed["reason"].str.replace(r" \(.*\)", "", regex=True).value_counts().rename("행 수").to_frame())
display(changed[["name", "driver", "lag", "n_v2", "rev_yoy_n", "verdict_v1", "pct_v1", "verdict_full200", "pct_full200",
                 "verdict_v2", "pct_v2", "reason"]]
        .rename(columns={"name": "기업", "driver": "드라이버", "lag": "선행", "n_v2": "표본", "rev_yoy_n": "매출 YoY 분기",
                         "verdict_v1": "v1 판정", "pct_v1": "v1 백분위(30개, 전체기간)",
                         "verdict_full200": "200개·전체기간 판정", "pct_full200": "200개·전체기간 백분위",
                         "verdict_v2": "v2 판정", "pct_v2": "v2 백분위(200개, 같은 분기)", "reason": "이유"}).round(1))

In [ ]:
# 표본이 짧은 드라이버: 위약 띠 폭이 어떻게 달라졌나 (시차 0)
short = all_v2[(all_v2["lag"] == 0) & (all_v2["n"] < all_v2["entity"].map(rev_n))]
band = short.merge(all_full[["entity", "driver", "lag", "placebo_p05", "placebo_p95"]],
                   on=["entity", "driver", "lag"], suffixes=("", "_full"))
band["띠 폭 v2(같은 분기)"] = band["placebo_p95"] - band["placebo_p05"]
band["띠 폭 전체기간"] = band["placebo_p95_full"] - band["placebo_p05_full"]
display(band[["name", "driver", "n", "corr", "띠 폭 전체기간", "띠 폭 v2(같은 분기)", "verdict"]]
        .rename(columns={"name": "기업", "driver": "드라이버", "n": "표본", "corr": "상관", "verdict": "v2 판정"}).round(3))

### 그림: 시차별 상관 (드라이버마다 자기 분기에서 구한 위약 200개의 5~95% 띠, 세로선 = 추세 상관)

In [ ]:
LABEL = {
    "trade:854232:expDlr": "메모리 수출액 854232", "trade:854231:expDlr": "프로세서 수출액 854231",
    "tw_revenue:2330": "TSMC 월매출", "tw_revenue:2408": "Nanya 월매출", "kr_peer:000660:revenue": "SK하이닉스 매출",
    "trade:842951:expDlr": "휠로더 수출액 842951", "trade:842952:expDlr": "굴착기 수출액 842952",
    "fred:HOUST": "미국 주택착공", "fred:PERMIT": "미국 건축허가",
    "amazon_bsr:median_rank": "아마존 순위 중앙값(낮을수록 잘 팔림)", "google_trends:US:medicube": "구글 트렌드 medicube",
    "google_trends:US:korean skincare": "구글 트렌드 korean skincare", "fred:RSHPCS": "미국 건강·뷰티 소매판매",
    "kosis:kosis_113bb45a04e152ed": "온라인쇼핑 화장품 거래액", "kosis:kosis_b85f8425a11ca86a": "화장품 소매판매지수",
    "tourism:D:100": "출국 내국인", "tourism:E:TOTAL": "방한 외국인 합계", "airline:intl_dep_sum": "인천공항 국적사 출발 여객",
}
ITEM_KOR = {"expDlr": "수출액", "expWgt": "수출중량", "impDlr": "수입액", "unit_price": "단가(USD/kg)"}
NAME_HS = {"030341": "참치", "030342": "참치", "030343": "참치", "030344": "참치",
           "330499": "기타 화장품", "330420": "눈화장품"}
def lab(d):
    if d in LABEL:
        return LABEL[d]
    parts = d.split(":")
    if parts[0] == "trade" and len(parts) == 3:
        return f"{NAME_HS.get(parts[1], '')} {parts[1]} {ITEM_KOR.get(parts[2], parts[2])}".strip()
    return d

def lag_figure(eid, res, max_rows=12):
    spec = [d for d in pa.SPEC_DRIVERS[eid] if d in set(res["driver"])]
    others = (res[~res["driver"].isin(spec)].assign(a=lambda d: d["corr"].abs())
              .groupby("driver")["a"].max().sort_values(ascending=False).index.tolist())
    drivers = (spec + others)[:max_rows]
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 0.46 * len(drivers) + 1.8), sharey=True, facecolor=SURFACE)
    for ax, k in zip(axes, pa.LAGS):
        r = res[res["lag"] == k].set_index("driver").reindex(drivers)
        y = np.arange(len(drivers))
        ax.barh(y, r["placebo_p95"] - r["placebo_p05"], left=r["placebo_p05"], height=0.82, color=BAND,
                zorder=1, label="위약 hs 200개 5~95% (같은 분기)")
        ax.barh(y, r["corr"], height=0.42, color=BLUE, zorder=2, label="실제 드라이버 상관")
        ax.scatter(r["trend_corr"], y, marker="|", s=170, color=INK, linewidths=2, zorder=3, label="추세 가짜 시계열 상관")
        for yi, (c, s, n) in enumerate(zip(r["corr"], r["strong"], r["n"])):
            if pd.notna(c):
                txt = f"n={int(n)}" + (" 강함" if s else "")
                ax.text(1.04, yi, txt, ha="left", va="center", fontsize=7.5, color=INK2)
        ax.axvline(0, color=INK2, linewidth=0.8)
        ax.set_xlim(-1, 1.42); ax.set_xticks([-1, -0.5, 0, 0.5, 1]); ax.set_facecolor(SURFACE)
        ax.axvline(1.0, color=GRID, linewidth=0.8)
        ax.set_title(["동행", "1분기 선행", "2분기 선행"][k], color=INK, fontsize=11)
        ax.grid(axis="x", color=GRID, linewidth=0.8); ax.set_axisbelow(True)
        for s in ("top", "right"): ax.spines[s].set_visible(False)
        ax.tick_params(colors=INK2)
    axes[0].set_yticks(np.arange(len(drivers))); axes[0].set_yticklabels([lab(d) for d in drivers], color=INK)
    axes[0].invert_yaxis()
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.0), fontsize=9)
    fig.suptitle(f"{PILOTS[eid]} ({eid}) 매출 YoY 와 드라이버 YoY 의 상관 (v2)", y=1.06, color=INK, fontsize=12)
    fig.tight_layout()
    return fig

for eid in PILOTS:
    fig = lag_figure(eid, res_v2[eid])
    fig.savefig(FIG_DIR / f"corr_lags_{eid}.png", dpi=130, bbox_inches="tight", facecolor=SURFACE)
    plt.show()

### 그림: 매출 YoY 와 가장 뚜렷한 후보 드라이버 YoY (표본 20분기 이상 우선)

In [ ]:
def line_figure(eid, mask_note=None):
    r = res_v2[eid]; yy = yoys[eid]
    cand = r[r["driver"].isin(pa.SPEC_DRIVERS[eid])].dropna(subset=["corr"])
    cand = cand.assign(a=cand["corr"].abs(), clear=(cand["verdict"] == "뚜렷"), big=(cand["n"] >= 20))
    top = cand.sort_values(["clear", "big", "placebo_pct_abs", "a"], ascending=False).iloc[0]
    d, k = top["driver"], int(top["lag"])
    x = yy[d].shift(k) * 100; y = yy["revenue"] * 100
    fig, ax = plt.subplots(figsize=(11, 3.6), facecolor=SURFACE)
    ax.plot(y.index, y, color=BLUE, linewidth=2, label="매출 YoY")
    ax.plot(x.index, x, color=ORANGE, linewidth=2, label=f"{lab(d)} YoY ({k}분기 선행)")
    both = pd.concat([x, y]).dropna()
    lo, hi = np.nanpercentile(both, 2), np.nanpercentile(both, 98); pad = (hi - lo) * 0.1
    ax.set_ylim(lo - pad, hi + pad)
    clipped = int(((both < lo - pad) | (both > hi + pad)).sum())
    ax.axhline(0, color=INK2, linewidth=0.8)
    ax.set_ylabel("전년 같은 분기 대비 %", color=INK2); ax.set_facecolor(SURFACE)
    ax.grid(axis="y", color=GRID); ax.set_axisbelow(True)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    ax.legend(frameon=False, loc="upper left", fontsize=9)
    note = (f"상관 {top['corr']:.2f}, 표본 {int(top['n'])}분기, 위약 백분위 {top['placebo_pct_abs']:.1f} "
            f"(같은 분기 {int(top['n_placebo'])}개), 추세 상관 {top['trend_corr']:.2f}")
    if clipped:
        note += f" / 축 범위 밖 {clipped}점 생략"
    if mask_note:
        note += f" / {mask_note}"
    ax.set_title(f"{PILOTS[eid]} ({eid})  {note}", color=INK, fontsize=9.5, loc="left")
    fig.tight_layout()
    return fig

for eid in PILOTS:
    fig = line_figure(eid, "2022 Q4 ~ 2023 Q4 는 합병으로 YoY 비정상" if eid == "006040" else None)
    fig.savefig(FIG_DIR / f"yoy_line_{eid}.png", dpi=130, bbox_inches="tight", facecolor=SURFACE)
    plt.show()

## 5. 동원산업: 합병 전후 (v2 방식)
합병 전: YoY 의 기준·비교 분기가 모두 합병 전 (~2022 Q3). 합병 후: 둘 다 합병 후 (2024 Q1~).
위약도 각 구간·드라이버의 같은 분기에서 계산한다.

In [ ]:
eid = "006040"; yy = yoys[eid]; idx = yy.index
masks = {"전체(합병 영향 포함)": pd.Series(True, index=idx),
         "합병 전 (~2022 Q3)": pd.Series(idx <= pa.DONGWON_PRE_END, index=idx),
         "합병 후 (2024 Q1~)": pd.Series(idx >= pa.DONGWON_POST_START, index=idx)}
dongwon = pd.concat([pa.evaluate(yy, pa.SPEC_DRIVERS[eid], plcs[eid], avails[eid], mask=m).assign(구간=name)
                     for name, m in masks.items()], ignore_index=True)
dongwon.to_csv(TAB_DIR / "dongwon_merger_split_v2.csv", index=False, encoding="utf-8-sig")
display(dongwon.pivot_table(index=["driver", "lag"], columns="구간", values="corr").round(3).rename(index=lab, level=0))
display(dongwon.pivot_table(index=["driver", "lag"], columns="구간", values="verdict", aggfunc="first").rename(index=lab, level=0))
display(dongwon.groupby("구간")[["n", "n_placebo"]].max())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.4), sharey=True, facecolor=SURFACE)
drv = [d for d in pa.SPEC_DRIVERS[eid] if d.endswith("expDlr")]
for ax, name in zip(axes, masks):
    sub = dongwon[(dongwon["구간"] == name) & dongwon["driver"].isin(drv)]
    y = np.arange(len(drv))
    for j, (k, col) in enumerate(zip(pa.LAGS, [BLUE, ORANGE, AQUA])):
        r = sub[sub["lag"] == k].set_index("driver").reindex(drv)
        yy_ = y + (j - 1) * 0.27
        ax.barh(yy_, r["placebo_p95"] - r["placebo_p05"], left=r["placebo_p05"], height=0.25, color=BAND, zorder=1,
                label="위약 5~95% (같은 분기)" if j == 0 else None)
        ax.barh(yy_, r["corr"], height=0.13, color=col, zorder=2, label=["동행", "1분기 선행", "2분기 선행"][k])
    ax.axvline(0, color=INK2, linewidth=0.8); ax.set_xlim(-1, 1); ax.set_facecolor(SURFACE)
    ax.set_title(f"{name}  (표본 {int(sub['n'].max())}분기)", color=INK, fontsize=10)
    ax.grid(axis="x", color=GRID); ax.set_axisbelow(True)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
axes[0].set_yticks(np.arange(len(drv))); axes[0].set_yticklabels([lab(d) for d in drv]); axes[0].invert_yaxis()
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.02), fontsize=9)
fig.suptitle("동원산업 매출 YoY 와 참치 수출액 YoY 의 상관: 합병 전후 (v2)", y=1.08, color=INK)
fig.tight_layout()
fig.savefig(FIG_DIR / "dongwon_merger_split.png", dpi=130, bbox_inches="tight", facecolor=SURFACE)
plt.show()

## 6. 하나투어: 코로나 구간(YoY 2020 Q1 ~ 2023 Q4) 제외 (v2 방식)

In [ ]:
eid = "039130"; yy = yoys[eid]; idx = yy.index
m = pd.Series((idx < "2020-03-31") | (idx > "2023-12-31"), index=idx)
hana = pa.evaluate(yy, pa.SPEC_DRIVERS[eid], plcs[eid], avails[eid], mask=m)
hana.to_csv(TAB_DIR / "hanatour_ex_covid_v2.csv", index=False, encoding="utf-8-sig")
show(hana)

## 7. 확인

In [ ]:
checks = pd.DataFrame([
    ("추세 상관이 드라이버와 같은 분기에서 계산됨 (trend_n = n)", bool((all_v2["trend_n"] == all_v2["n"]).all())),
    ("위약 200개를 채운 행 비율", f"{(all_v2['n_placebo'] == pa.N_PLACEBO).mean():.1%}"),
    ("위약 200개 미만 행 (최소 수)", f"{int((all_v2['n_placebo'] < pa.N_PLACEBO).sum())}행 (최소 {int(all_v2['n_placebo'].min())})"),
    ("as_of 미래 값", int(asof_check["미래 값"].sum())),
    ("이 노트북 실행 중 DB 쓰기", len(db.WRITE_LOG) - WRITES_AT_START),
    ("그림 수", len(list(FIG_DIR.glob("*.png")))),
    ("표 수", len(list(TAB_DIR.glob("*.csv")))),
], columns=["항목", "결과"])
display(checks)